# Supervised Learning (Advanced): Regularization and Ensemble Methods

This notebook implements the assignment requirements:
1. Regularization on the diabetes dataset (`LinearRegression`, `Ridge`, `Lasso`) with MSE comparison.
2. Ensemble/classification methods on the breast cancer dataset (`DecisionTreeClassifier`, `RandomForestClassifier`, `GradientBoostingClassifier`) with F1 and AUC comparison.

## 1) Setup and Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import mean_squared_error, f1_score, roc_auc_score

RANDOM_STATE = 42

## 2) Load and Inspect Diabetes Dataset

In [2]:
diabetes = load_diabetes()
X_diabetes = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_diabetes = pd.Series(diabetes.target, name="target")

print("Diabetes feature shape:", X_diabetes.shape)
print("Diabetes target shape:", y_diabetes.shape)
print("\nFeature names:", list(X_diabetes.columns))
print("\nSummary statistics:")
display(X_diabetes.describe().T)

Diabetes feature shape: (442, 10)
Diabetes target shape: (442,)

Feature names: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']

Summary statistics:


,count,mean,std,min,25%,50%,75%,max
age,442.0,-2.511817e-19,0.047619,-0.107226,-0.037299,0.005383,0.038076,0.110727
sex,442.0,1.230790e-17,0.047619,-0.044642,-0.044642,-0.044642,0.050680,0.050680
bmi,442.0,-2.245564e-16,0.047619,-0.090275,-0.034229,-0.007284,0.031248,0.170555
bp,442.0,-4.797570e-17,0.047619,-0.112399,-0.036656,-0.005670,0.035644,0.132044
s1,442.0,-1.381499e-17,0.047619,-0.126781,-0.034248,-0.004321,0.028358,0.153914
s2,442.0,3.918434e-17,0.047619,-0.115613,-0.030358,-0.003819,0.029844,0.198788
s3,442.0,-5.777179e-18,0.047619,-0.102307,-0.035117,-0.006584,0.029312,0.181179
s4,442.0,-9.042540e-18,0.047619,-0.076395,-0.039493,-0.002592,0.034309,0.185234
s5,442.0,9.268604e-17,0.047619,-0.126097,-0.033246,-0.001947,0.032432,0.133597
s6,442.0,1.130318e-17,0.047619,-0.137767,-0.033179,-0.001078,0.027917,0.135612


## 3) Train/Test Split for Regression

In [3]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=RANDOM_STATE
)

print("X_train_reg:", X_train_reg.shape)
print("X_test_reg :", X_test_reg.shape)

X_train_reg: (353, 10)
X_test_reg : (89, 10)


## 4) Baseline Linear Regression (MSE)

In [9]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_reg, y_train_reg)

y_pred_lin = lin_reg.predict(X_test_reg)
mse_lin = mean_squared_error(y_test_reg, y_pred_lin)

print(f"Linear Regression Test MSE: {mse_lin:.4f}")

Linear Regression Test MSE: 2900.1936


## 5) Ridge Regression with GridSearchCV

In [6]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(random_state=RANDOM_STATE))
])

alpha_grid = np.logspace(-4, 2, 20)

ridge_param_grid = {
    "model__alpha": alpha_grid
}

ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

ridge_grid.fit(X_train_reg, y_train_reg)

best_ridge = ridge_grid.best_estimator_
y_pred_ridge = best_ridge.predict(X_test_reg)
mse_ridge = mean_squared_error(y_test_reg, y_pred_ridge)

print("Best Ridge params:", ridge_grid.best_params_)
print(f"Best Ridge CV MSE: {-ridge_grid.best_score_:.4f}")
print(f"Ridge Test MSE   : {mse_ridge:.4f}")

Best Ridge params: {'model__alpha': 23.357214690901213}
Best Ridge CV MSE: 3121.6801
Ridge Test MSE   : 2865.8417


## 6) Lasso Regression with GridSearchCV

In [7]:
lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(max_iter=10000, random_state=RANDOM_STATE))
])

lasso_param_grid = {
    "model__alpha": alpha_grid
}

lasso_grid = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=lasso_param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

lasso_grid.fit(X_train_reg, y_train_reg)

best_lasso = lasso_grid.best_estimator_
y_pred_lasso = best_lasso.predict(X_test_reg)
mse_lasso = mean_squared_error(y_test_reg, y_pred_lasso)

print("Best Lasso params:", lasso_grid.best_params_)
print(f"Best Lasso CV MSE: {-lasso_grid.best_score_:.4f}")
print(f"Lasso Test MSE   : {mse_lasso:.4f}")

Best Lasso params: {'model__alpha': 1.2742749857031321}
Best Lasso CV MSE: 3127.8499
Lasso Test MSE   : 2810.6340


## 7) Regression Model Comparison Table

In [8]:
regression_results = pd.DataFrame([
    {
        "Model": "LinearRegression",
        "Best_Params": {},
        "CV_Best_MSE": np.nan,
        "Test_MSE": mse_lin
    },
    {
        "Model": "Ridge (tuned)",
        "Best_Params": ridge_grid.best_params_,
        "CV_Best_MSE": -ridge_grid.best_score_,
        "Test_MSE": mse_ridge
    },
    {
        "Model": "Lasso (tuned)",
        "Best_Params": lasso_grid.best_params_,
        "CV_Best_MSE": -lasso_grid.best_score_,
        "Test_MSE": mse_lasso
    }
]).sort_values("Test_MSE", ascending=True).reset_index(drop=True)

display(regression_results)

,Model,Best_Params,CV_Best_MSE,Test_MSE
0,Lasso (tuned),{'model__alpha': 1.2742749857031321},3127.849930,2810.634028
1,Ridge (tuned),{'model__alpha': 23.357214690901213},3121.680062,2865.841740
2,LinearRegression,{},NaN,2900.193628


### Question 1 - Regression Results Analysis

The table above compares three regression models on the diabetes dataset, ranked by Test MSE (lower is better):

1. **Lasso (tuned)** achieved the lowest Test MSE (2810.63). GridSearchCV selected an optimal alpha of ~1.27, which applies L1 regularization to shrink less important feature coefficients to exactly zero — effectively performing feature selection while reducing overfitting.

2. **Ridge (tuned)** came in second (Test MSE 2865.84). Its optimal alpha of ~23.36, found via GridSearchCV, applies L2 regularization to shrink all coefficients toward zero without eliminating any, which reduces variance at the cost of slight bias.

3. **LinearRegression** (no regularization) had the highest Test MSE (2900.19), showing that even moderate regularization improves generalization on this dataset.

**Key takeaway:** Both Ridge and Lasso outperform plain Linear Regression, confirming that regularization helps reduce overfitting. Lasso's feature selection capability gives it a slight edge here, suggesting that some features in the diabetes dataset contribute more noise than signal.

## 8) Load and Inspect Breast Cancer Dataset

In [10]:
breast_cancer = load_breast_cancer()
X_cancer = pd.DataFrame(breast_cancer.data, columns=breast_cancer.feature_names)
y_cancer = pd.Series(breast_cancer.target, name="target")

print("Breast cancer feature shape:", X_cancer.shape)
print("Breast cancer target shape:", y_cancer.shape)
print("\nClass distribution:")
print(y_cancer.value_counts(normalize=True).rename("proportion"))

Breast cancer feature shape: (569, 30)
Breast cancer target shape: (569,)

Class distribution:
target
1    0.627417
0    0.372583
Name: proportion, dtype: float64


## 9) Train/Test Split for Classification

In [11]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_cancer,
    y_cancer,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_cancer
)

print("X_train_clf:", X_train_clf.shape)
print("X_test_clf :", X_test_clf.shape)

X_train_clf: (455, 30)
X_test_clf : (114, 30)


## 10) Decision Tree with GridSearchCV

In [12]:
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)

dt_param_grid = {
    "max_depth": [None, 3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy"]
}

dt_grid = GridSearchCV(
    estimator=dt,
    param_grid=dt_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

dt_grid.fit(X_train_clf, y_train_clf)
best_dt = dt_grid.best_estimator_

y_pred_dt = best_dt.predict(X_test_clf)
y_proba_dt = best_dt.predict_proba(X_test_clf)[:, 1]

f1_dt = f1_score(y_test_clf, y_pred_dt)
auc_dt = roc_auc_score(y_test_clf, y_proba_dt)

print("Best Decision Tree params:", dt_grid.best_params_)
print(f"Best Decision Tree CV F1: {dt_grid.best_score_:.4f}")
print(f"Decision Tree Test F1   : {f1_dt:.4f}")
print(f"Decision Tree Test AUC  : {auc_dt:.4f}")

Best Decision Tree params: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 5}
Best Decision Tree CV F1: 0.9518
Decision Tree Test F1   : 0.9362
Decision Tree Test AUC  : 0.9163


## 11) Random Forest with GridSearchCV

In [13]:
rf = RandomForestClassifier(random_state=RANDOM_STATE)

rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

rf_grid.fit(X_train_clf, y_train_clf)
best_rf = rf_grid.best_estimator_

y_pred_rf = best_rf.predict(X_test_clf)
y_proba_rf = best_rf.predict_proba(X_test_clf)[:, 1]

f1_rf = f1_score(y_test_clf, y_pred_rf)
auc_rf = roc_auc_score(y_test_clf, y_proba_rf)

print("Best Random Forest params:", rf_grid.best_params_)
print(f"Best Random Forest CV F1: {rf_grid.best_score_:.4f}")
print(f"Random Forest Test F1   : {f1_rf:.4f}")
print(f"Random Forest Test AUC  : {auc_rf:.4f}")

Best Random Forest params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best Random Forest CV F1: 0.9684
Random Forest Test F1   : 0.9655
Random Forest Test AUC  : 0.9931


## 12) Gradient Boosting with GridSearchCV

In [14]:
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)

gb_param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0]
}

gb_grid = GridSearchCV(
    estimator=gb,
    param_grid=gb_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

gb_grid.fit(X_train_clf, y_train_clf)
best_gb = gb_grid.best_estimator_

y_pred_gb = best_gb.predict(X_test_clf)
y_proba_gb = best_gb.predict_proba(X_test_clf)[:, 1]

f1_gb = f1_score(y_test_clf, y_pred_gb)
auc_gb = roc_auc_score(y_test_clf, y_proba_gb)

print("Best Gradient Boosting params:", gb_grid.best_params_)
print(f"Best Gradient Boosting CV F1: {gb_grid.best_score_:.4f}")
print(f"Gradient Boosting Test F1    : {f1_gb:.4f}")
print(f"Gradient Boosting Test AUC   : {auc_gb:.4f}")

Best Gradient Boosting params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Best Gradient Boosting CV F1: 0.9810
Gradient Boosting Test F1    : 0.9660
Gradient Boosting Test AUC   : 0.9937


## 13) Classification Evaluation (F1, AUC)

In [15]:
classification_metrics = pd.DataFrame([
    {"Model": "DecisionTreeClassifier", "Test_F1": f1_dt, "Test_AUC": auc_dt},
    {"Model": "RandomForestClassifier", "Test_F1": f1_rf, "Test_AUC": auc_rf},
    {"Model": "GradientBoostingClassifier", "Test_F1": f1_gb, "Test_AUC": auc_gb},
]).sort_values(["Test_AUC", "Test_F1"], ascending=False).reset_index(drop=True)

display(classification_metrics)

,Model,Test_F1,Test_AUC
0,GradientBoostingClassifier,0.965986,0.993717
1,RandomForestClassifier,0.965517,0.993056
2,DecisionTreeClassifier,0.936170,0.916336


## 14) Ensemble Model Comparison Table

In [16]:
ensemble_results = pd.DataFrame([
    {
        "Model": "DecisionTreeClassifier",
        "Best_Params": dt_grid.best_params_,
        "CV_Best_F1": dt_grid.best_score_,
        "Test_F1": f1_dt,
        "Test_AUC": auc_dt
    },
    {
        "Model": "RandomForestClassifier",
        "Best_Params": rf_grid.best_params_,
        "CV_Best_F1": rf_grid.best_score_,
        "Test_F1": f1_rf,
        "Test_AUC": auc_rf
    },
    {
        "Model": "GradientBoostingClassifier",
        "Best_Params": gb_grid.best_params_,
        "CV_Best_F1": gb_grid.best_score_,
        "Test_F1": f1_gb,
        "Test_AUC": auc_gb
    }
]).sort_values(["Test_AUC", "Test_F1"], ascending=False).reset_index(drop=True)

display(ensemble_results)

,Model,Best_Params,CV_Best_F1,Test_F1,Test_AUC
0,GradientBoostingClassifier,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti...",0.980987,0.965986,0.993717
1,RandomForestClassifier,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.968444,0.965517,0.993056
2,DecisionTreeClassifier,"{'criterion': 'gini', 'max_depth': 5, 'min_sam...",0.951770,0.936170,0.916336


### Question 2 - Classification Results Analysis

#### 1. Performance Comparison (F1 Score and AUC)

| Model | Test F1 | Test AUC |
|-------|---------|----------|
| GradientBoostingClassifier | 0.9660 | 0.9937 |
| RandomForestClassifier | 0.9655 | 0.9931 |
| DecisionTreeClassifier | 0.9362 | 0.9163 |

- **GradientBoostingClassifier** achieved the best performance on both metrics (F1 = 0.9660, AUC = 0.9937), narrowly outperforming Random Forest. Its sequential boosting approach — where each new tree corrects the errors of previous ones — allows it to learn complex patterns more effectively.
- **RandomForestClassifier** performed nearly as well (F1 = 0.9655, AUC = 0.9931). By averaging predictions across many independent trees (bagging), it achieves strong generalization with reduced variance compared to a single tree.
- **DecisionTreeClassifier** lagged behind both ensemble methods (F1 = 0.9362, AUC = 0.9163). As a single tree, it is more prone to overfitting and lacks the variance reduction that comes from aggregating multiple learners.

The gap between the two ensemble methods and the standalone Decision Tree highlights the core benefit of ensemble learning: combining multiple weak learners yields substantially better and more robust predictions.

#### 2. Hyperparameter Tuning via GridSearchCV

All three classifiers were tuned using **GridSearchCV with 5-fold cross-validation**, optimizing for F1 score:

- **DecisionTreeClassifier** — Best params: `criterion='gini'`, `max_depth=5`, `min_samples_split=5`, `min_samples_leaf=1` (CV F1 = 0.9518). Limiting `max_depth` to 5 prevented overfitting, and a slightly higher `min_samples_split` of 5 helped the tree generalize better by avoiding splits on very small subsets.

- **RandomForestClassifier** — Best params: `n_estimators=200`, `max_depth=None`, `max_features='sqrt'`, `min_samples_split=2`, `min_samples_leaf=1` (CV F1 = 0.9684). Unrestricted depth works well here because bagging across 200 trees naturally controls variance. Using `sqrt` of the features at each split introduces diversity among trees, strengthening the ensemble.

- **GradientBoostingClassifier** — Best params: `n_estimators=200`, `learning_rate=0.1`, `max_depth=3`, `subsample=0.8` (CV F1 = 0.9810). Shallow trees (`max_depth=3`) act as weak learners that the boosting process combines sequentially. The `subsample=0.8` introduces stochastic gradient boosting, adding regularization, while `learning_rate=0.1` balances learning speed with generalization.

## 15) Reference and Attribution Cell

### References
- scikit-learn documentation: https://scikit-learn.org/stable/
- Diabetes dataset API: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html
- Breast cancer dataset API: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html
- GridSearchCV API: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

### Attribution
- No classmate work was referenced.